# 00 Run the full reproducibility pipeline

This notebook runs the same canonical scripts used by the repository pipeline, in order. It writes notebook-generated artifacts to separate directories so they can be compared with the original command-line script outputs.

Script/reference artifacts remain in `data/processed/`, `outputs/`, and `figures/`. Notebook artifacts are written to `data/processed_notebooks/`, `outputs_notebooks/`, and `figures_notebooks/`. Comparison manifests are written to `outputs_comparison/`.

## Execution order

1. `scripts/01_prepare_data.R`
2. `scripts/02_fixed_effects_models.R`
3. `scripts/03_olse_cfa_measurement.R`
4. `scripts/04_sem_mechanism_model.R`
5. `scripts/05_make_paper_tables.R`
6. `scripts/06_make_figures.R`

In [ ]:
# Notebook setup: make execution robust from either repo root or notebooks/
find_project_root <- function(start = getwd()) {
  current <- normalizePath(start, winslash = "/", mustWork = FALSE)
  repeat {
    has_markers <- file.exists(file.path(current, "data")) &&
      file.exists(file.path(current, "scripts")) &&
      file.exists(file.path(current, "R"))
    if (has_markers) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("Could not find repository root.", call. = FALSE)
    current <- parent
  }
}

PROJECT_ROOT <- find_project_root(getwd())
setwd(PROJECT_ROOT)
message("Project root: ", PROJECT_ROOT)

# Reference locations used by the command-line R scripts.
SCRIPT_DATA_PROCESSED <- file.path(PROJECT_ROOT, "data", "processed")
SCRIPT_OUTPUT_ROOT <- file.path(PROJECT_ROOT, "outputs")
SCRIPT_FIGURES_DIR <- file.path(PROJECT_ROOT, "figures")

# Notebook-generated artifacts are intentionally written separately so reviewers
# can compare notebook outputs against the original script outputs.
NOTEBOOK_DATA_PROCESSED <- file.path(PROJECT_ROOT, "data", "processed_notebooks")
NOTEBOOK_OUTPUT_ROOT <- file.path(PROJECT_ROOT, "outputs_notebooks")
NOTEBOOK_FIGURES_DIR <- file.path(PROJECT_ROOT, "figures_notebooks")
COMPARISON_ROOT <- file.path(PROJECT_ROOT, "outputs_comparison")

Sys.setenv(
  LENS_DATA_PROCESSED_DIR = NOTEBOOK_DATA_PROCESSED,
  LENS_OUTPUT_DIR = NOTEBOOK_OUTPUT_ROOT,
  LENS_FIGURES_DIR = NOTEBOOK_FIGURES_DIR
)

source(file.path("R", "00_packages.R"))
source(file.path("R", "00_paths.R"))
source(file.path("R", "utils_io.R"))

preview_csv <- function(path, n = 8) {
  if (!file.exists(path)) stop("Missing expected file: ", path, call. = FALSE)
  readr::read_csv(path, show_col_types = FALSE) |>
    dplyr::slice_head(n = n)
}

assert_files_exist <- function(paths) {
  missing <- paths[!file.exists(paths)]
  if (length(missing) > 0) {
    stop("Missing expected output file(s):\n", paste(missing, collapse = "\n"), call. = FALSE)
  }
  tibble::tibble(file = paths, size_bytes = file.info(paths)$size)
}

list_relative_files <- function(base_dir, pattern = "\\.(csv|html|xlsx|png)$") {
  if (!dir.exists(base_dir)) return(character())
  files <- list.files(base_dir, pattern = pattern, recursive = TRUE, full.names = TRUE)
  sort(sub(paste0("^", normalizePath(base_dir, winslash = "/", mustWork = FALSE), "/?"), "", normalizePath(files, winslash = "/", mustWork = FALSE)))
}

md5_or_na <- function(path) {
  if (!file.exists(path)) return(NA_character_)
  unname(as.character(tools::md5sum(path)))
}

compare_file_sets <- function(script_base, notebook_base, group, pattern = "\\.(csv|html|xlsx|png)$") {
  script_rel <- list_relative_files(script_base, pattern)
  notebook_rel <- list_relative_files(notebook_base, pattern)
  rel_paths <- sort(unique(c(script_rel, notebook_rel)))
  if (length(rel_paths) == 0) {
    return(tibble::tibble(
      group = character(), relative_path = character(), script_path = character(), notebook_path = character(),
      script_exists = logical(), notebook_exists = logical(), script_md5 = character(), notebook_md5 = character(), status = character()
    ))
  }
  out <- tibble::tibble(
    group = group,
    relative_path = rel_paths,
    script_path = file.path(script_base, rel_paths),
    notebook_path = file.path(notebook_base, rel_paths)
  ) |>
    dplyr::mutate(
      script_exists = file.exists(script_path),
      notebook_exists = file.exists(notebook_path),
      script_md5 = vapply(script_path, md5_or_na, character(1)),
      notebook_md5 = vapply(notebook_path, md5_or_na, character(1)),
      status = dplyr::case_when(
        !script_exists ~ "missing_script_reference",
        !notebook_exists ~ "missing_notebook_output",
        script_md5 == notebook_md5 ~ "match",
        TRUE ~ "different"
      )
    )
  out
}

write_comparison_manifest <- function(filename = "notebook_vs_script_manifest.csv") {
  dir.create(COMPARISON_ROOT, recursive = TRUE, showWarnings = FALSE)
  manifest <- dplyr::bind_rows(
    compare_file_sets(SCRIPT_DATA_PROCESSED, DATA_PROCESSED, "processed_data", "\\.csv$"),
    compare_file_sets(SCRIPT_OUTPUT_ROOT, OUTPUT_ROOT, "analysis_outputs", "\\.(csv|html|xlsx)$"),
    compare_file_sets(SCRIPT_FIGURES_DIR, OUT_FIGURES, "figures", "\\.(csv|png)$")
  )
  out_path <- file.path(COMPARISON_ROOT, filename)
  readr::write_csv(manifest, out_path, na = "")
  message("Wrote comparison manifest: ", out_path)
  manifest
}

message("Notebook processed data directory: ", DATA_PROCESSED)
message("Notebook output directory: ", OUTPUT_ROOT)
message("Notebook figures directory: ", OUT_FIGURES)
message("Comparison manifests directory: ", COMPARISON_ROOT)

In [ ]:
pipeline_scripts <- c(
  "scripts/01_prepare_data.R",
  "scripts/02_fixed_effects_models.R",
  "scripts/03_olse_cfa_measurement.R",
  "scripts/04_sem_mechanism_model.R",
  "scripts/05_make_paper_tables.R",
  "scripts/06_make_figures.R"
)

for (script in pipeline_scripts) {
  message("\n--- Running ", script, " ---")
  # Source into the global notebook session so each script's internal
  # source("R/00_paths.R") call sees the notebook output-path environment
  # variables and does not fall back to repo-default outputs/.
  source(script, local = globalenv())
}


## Notebook output audit

In [ ]:
key_outputs <- c(
  file.path(DATA_PROCESSED, "student_term_fe_analysis.csv"),
  file.path(DATA_PROCESSED, "olse_scored_for_sem.csv"),
  file.path(OUT_FE, "project_soar_fe_grade_coefficients.csv"),
  file.path(OUT_OLSE, "olse_cfa_fit_comparison.csv"),
  file.path(OUT_SEM, "sem_primary_params.csv"),
  file.path(OUT_TABLES, "lens_paper_tables.xlsx"),
  file.path(OUT_FIGURES, "figure_inventory.csv")
)

assert_files_exist(key_outputs)

## Script-vs-notebook comparison manifest

This final cell recomputes the comparison manifest from the actual files currently on disk.

In [ ]:
manifest <- write_comparison_manifest("00_full_pipeline_comparison.csv")

manifest_summary <- manifest |>
  dplyr::count(status, name = "n") |>
  dplyr::arrange(status)

manifest_summary
